# Fine-tune Qwen2.5-Coder-0.5B on the Dockerfile/YAML/HCL corpus (Kaggle T4 x2, ~30GB)

LoRA fine-tuning pipeline for **`Qwen/Qwen2.5-Coder-0.5B`**, trained on the *same* corpus and
language slice as `docker_auto_complete`'s from-scratch 80M model
(`~/PROJECTS/LLM_FROM_SCRATCH/PIPELINE/docker-model-scratch.ipynb`): `bigcode/the-stack-dedup`,
interleaved over `dockerfile` / `yaml` / `hcl`.

Why LoRA instead of a full fine-tune: Qwen2.5-Coder-0.5B is already pretrained on a huge code
corpus, so this is domain adaptation, not teaching it syntax from zero. LoRA gets there with far
less compute/memory than full fine-tuning, and checkpoints are a few MB instead of ~1GB+.

**Structural choices carried over from the original notebook, on purpose (same author, same
constraints):**
- Chunked/resumable rounds — a Kaggle session can be closed and reopened between rounds, same
  as the original pipeline. Round 0 = Phase 1 (fresh LoRA adapter), every round after = Phase 2
  (resume adapter + optimizer state from the previous round's checkpoint).
- One continuous cosine LR schedule spanning all rounds (no re-warmup at round boundaries).
- Same dataset, same languages, same streaming/`.skip()` approach.

**One real bug fixed relative to the original pipeline, not just copied over:** the original
notebook packs token ids as `uint16` because GPT-2 BPE's vocab (50257) just barely fits. Qwen2.5's
tokenizer has a **151,936**-token vocab — `uint16` maxes out at 65,535, so it would silently wrap
around and corrupt every id above that. This notebook uses `uint32` for token ids throughout.

**This version actually uses both T4s, launched as a real subprocess, not a fork.** An earlier
version of this notebook drove both GPUs via `accelerate.notebook_launcher`, which forks worker
processes from the Jupyter kernel. On Kaggle's image that failed with
`Cannot re-initialize CUDA in forked subprocess`: something in the parent kernel touches CUDA
before the fork happens (exactly what triggers it varies by image/driver/torch version and isn't
worth chasing), and the forked children inherit a broken CUDA context. The fix: the training loop
now lives in a standalone script (`train_worker.py`, generated by the cell below from your current
config) and is launched with `accelerate launch` as a genuine separate OS process tree via
`torchrun` — completely sidestepping fork/CUDA-init interaction, and the standard way multi-GPU
training is actually run from notebooks in practice.

The batch/context-length knobs are sized to actually use most of each card's memory rather than
leaving headroom on the table. See **"Sizing knobs to your VRAM"** below before your first run —
the shipped defaults are a reasonable starting point, not a promise; watch the printed per-GPU
memory line from step 0 of round 0 and adjust from there.

**One-time setup required (same as the original notebook):** `bigcode/the-stack-dedup` is a gated
dataset — accept its terms on the HF dataset page, generate an HF token, and add it as a Kaggle
secret named `HF_TOKEN` (Add-ons -> Secrets) before running the cells below. `Qwen/Qwen2.5-Coder-0.5B`
itself is Apache-2.0 and NOT gated, so no separate approval is needed for the model.

**Kaggle GPU setting:** use the `GPU T4 x2` accelerator (Settings -> Accelerator).


In [ ]:
!pip install -q -U transformers accelerate peft datasets huggingface_hub
# Kaggle's preinstalled torchao (often an old build) can be older than what a fresh `peft`
# requires -- peft's LoRA dispatcher raises on that version mismatch even though this
# pipeline never uses torchao (no quantization here). Nothing depends on it, so drop it.
!pip uninstall -y -q torchao


In [ ]:
import os

# bigcode/the-stack-dedup is gated: accept its terms on the HF dataset page first,
# then add a Hugging Face token as a Kaggle secret named "HF_TOKEN" (Add-ons -> Secrets).
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass  # not running on Kaggle, or the secret isn't set -- falls back to any cached CLI login

if os.environ.get("HF_TOKEN"):
    from huggingface_hub import login
    login(token=os.environ["HF_TOKEN"])
else:
    print("No HF_TOKEN found -- set it as a Kaggle secret before running the tokenize/train cells, "
          "or the-stack-dedup will fail to load.")


In [ ]:
import os
import gc
import math
import shutil
import subprocess
import numpy as np
import torch

# ---- Base model / dataset ----
MODEL_NAME    = "Qwen/Qwen2.5-Coder-0.5B"     # base (not -Instruct): we want raw next-token completion
DATASET_NAME  = "bigcode/the-stack-dedup"     # same corpus as the from-scratch 80M model
IAC_LANGUAGES = ["dockerfile", "yaml", "hcl"] # same language slice: Dockerfiles, CI/K8s configs, Terraform

# Reduces allocator fragmentation-driven near-miss OOMs (see the PyTorch OOM message
# suggesting this) -- set once here, inherited by the accelerate-launched subprocess
# and its child training processes.
os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")

NUM_GPUS = torch.cuda.device_count()
print(f"GPUs visible: {NUM_GPUS}")
if NUM_GPUS < 2:
    print("WARNING: expected 2 GPUs (Kaggle 'GPU T4 x2'). Training will still work with 1, "
          "just without the second card's throughput/memory.")

DATA_DIR  = "/kaggle/working/chunks"            # one .bin file per round (regenerated each session)
CKPT_DIR  = "/kaggle/working/lora_checkpoints"
ADAPTER_DIR       = os.path.join(CKPT_DIR, "adapter")       # PEFT adapter weights (small, a few MB-tens of MB)
TRAIN_STATE_PATH  = os.path.join(CKPT_DIR, "train_state.pt")  # optimizer state + global_step + round_idx
TRAIN_SCRIPT_PATH = "/kaggle/working/train_worker.py"        # generated by the cell below, from current config

# Same read-only-input-mount pattern as the original notebook: point this at your uploaded
# Kaggle Dataset containing a previous session's `adapter/` + `train_state.pt`, then the
# Phase-2 bootstrap cell below copies it into the writable CKPT_DIR.
KAGGLE_INPUT_CKPT_DIR = "/kaggle/input/qwen-dockerfile-lora-checkpoints"  # replace with your actual input path


## Sizing knobs to your VRAM

**Read this before touching `BATCH_SIZE`/`BLOCK_SIZE` -- Qwen's vocab changes the math.**
The first version of these defaults (`BLOCK_SIZE=2048`, `BATCH_SIZE=16`) OOM'd immediately, and not
on the usual "activations got too big" grounds. Qwen2.5's tokenizer has a **151,936**-token vocab
(3x GPT-2's ~50k), so the `logits` tensor from a forward pass is `[BATCH_SIZE, BLOCK_SIZE, 151936]`
-- at the old defaults that's already **~10GB in fp16**, and the loss function upcasts it to fp32
for numerical stability, needing **another ~20GB** on top, transiently. That swamps a 14.56GB-usable
T4 regardless of how small the 0.5B model itself is. This is a fundamentally different memory
profile than the from-scratch model's GPT-2 vocab (50257) -- **the vocab-size term dominates memory
here far more than the usual batch/block-size activation cost does**, and it hits during the very
first eval at step 0, before you ever see a per-step training log.

Rule of thumb used for the new defaults below: keep `BATCH_SIZE * BLOCK_SIZE` under roughly
**8,000-10,000** on a 14.56GB T4 (`fp16_logits + fp32_logits sized tensors coexisting transiently`
is the constraint: `BATCH_SIZE * BLOCK_SIZE * VOCAB_SIZE * 6 bytes` should stay well under what's
left after the ~1GB base model and other activations). The shipped defaults
(`BLOCK_SIZE=1024, BATCH_SIZE=4`, product 4096) leave real headroom; scale up cautiously.

- **`BLOCK_SIZE`** (sequence length) and **`BATCH_SIZE`** (per-GPU, per-microbatch) are still the
  two biggest levers, but their *product* is now the thing to watch, because of the vocab-size
  interaction above -- not just each one independently.
- **`GRAD_ACCUM_STEPS`** grows the *effective* batch size (`BATCH_SIZE * NUM_GPUS * GRAD_ACCUM_STEPS`)
  without any extra peak memory cost -- raise this instead of `BATCH_SIZE` once you're near the
  `BATCH_SIZE * BLOCK_SIZE` ceiling above but still want a bigger effective batch.
- **`USE_GRADIENT_CHECKPOINTING`** trades compute for memory on the *non-logits* activations
  (attention/MLP). It's **on by default** now -- a training step needs to *retain* every layer's
  activations for backward, unlike the step-0 eval pass (which runs under `torch.no_grad()` and
  frees them immediately). At this pipeline's batch/block settings, that gap is exactly what
  pushed the first real training step over a 14.56GB T4's limit even though the eval step right
  before it succeeded comfortably -- checkpointing recomputes activations in the backward pass
  instead of holding all 24 layers' worth at once, which is what makes the difference here.
  A useful mental model going forward: **eval succeeding tells you almost nothing about whether
  a training step will fit** -- size your batch/block against a training step, not the eval log.
- **`LORA_R`** barely moves memory (LoRA adapters are tiny relative to the base 0.5B model) -- raise
  it for capacity/quality, not to consume VRAM.

**Whenever you change any of these, re-run the "generate train_worker.py" cell below before your
next Phase 1/Phase 2 run** -- the script is a snapshot of this config at write time, it does not
read these variables live.

**How to tune it in practice:** run Phase 1 with the shipped (conservative) defaults first, and
let it get through the step-0 eval and first training step -- that's the point where the old
defaults died. Once you see the `[rank 0] GPU mem allocated / reserved` and `[rank 1] ...` lines,
raise `BATCH_SIZE` by small increments (e.g. +2) and rerun, watching that line, rather than jumping
straight back to a large value -- the backward pass on the next training step needs more memory
than the forward-only eval pass that crashed last time, so "eval survived" doesn't by itself mean
a training step will too.

In [ ]:
# ---- Chunked training plan ----
# Starting guesses, same spirit as the original notebook: retune LINES_PER_CHUNK / STEPS_PER_ROUND
# after round 0 once you've seen actual tokens-per-file and it/s on your GPUs. LoRA fine-tuning a
# pretrained model needs *far* fewer total steps than training from scratch -- this plan targets a
# few passes over a moderate slice of the corpus, not the full multi-week schedule of the original.
LINES_PER_CHUNK = 20_000     # dataset rows (files) consumed per round
NUM_ROUNDS      = 5          # round 0 = Phase 1, rounds 1..NUM_ROUNDS-1 = Phase 2
STEPS_PER_ROUND = 2_000      # gradient steps trained in each round -- RETUNE after round 0
TOTAL_STEPS     = NUM_ROUNDS * STEPS_PER_ROUND   # one continuous LR schedule across all rounds

# ---- Packing / batching (see "Sizing knobs to your VRAM" above -- Qwen's ~152k vocab means
# BATCH_SIZE * BLOCK_SIZE is the ceiling to watch, not either knob alone) ----
BLOCK_SIZE                 = 1024    # sequence length per training example
BATCH_SIZE                 = 4       # per GPU, per microbatch -- scale up cautiously, see markdown above
GRAD_ACCUM_STEPS           = 8       # effective global batch = BATCH_SIZE * NUM_GPUS * GRAD_ACCUM_STEPS = 64
USE_GRADIENT_CHECKPOINTING = True    # ON by default -- see markdown above: training retains every
                                      # layer's activations for backward (eval does not), and at
                                      # these batch/block settings that tips a 14.56GB T4 into OOM
                                      # without it

# ---- LoRA hyperparameters ----
LORA_R          = 16
LORA_ALPHA      = 32
LORA_DROPOUT    = 0.05
LORA_TARGETS    = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

# ---- Optimization ----
warmup_iters  = 200
max_lr        = 2e-4     # LoRA tolerates a higher LR than full fine-tuning; base weights are frozen
min_lr        = 2e-5
eval_interval = 500       # also the mid-round checkpoint cadence
eval_batches  = 20

effective_batch = BATCH_SIZE * max(NUM_GPUS, 1) * GRAD_ACCUM_STEPS
logits_budget_elems = BATCH_SIZE * BLOCK_SIZE  # keep this under ~8,000-10,000 for a 14.56GB T4 at vocab~152k
print(f"per-GPU microbatch={BATCH_SIZE}, block_size={BLOCK_SIZE}, "
      f"batch*block={logits_budget_elems} (logits-memory ceiling to watch), "
      f"effective global batch={effective_batch}")


## Tokenizer (CPU only, parent process)

Only the tokenizer is loaded here. All dataset streaming/tokenization below runs on CPU in the
parent process; the model itself is only ever constructed inside `train_worker.py` (spawned as a
subprocess per training round) or in the single-GPU post-training cells at the very end.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

VOCAB_SIZE = len(tokenizer)
print("vocab size:", VOCAB_SIZE)


## Streaming the dataset (same corpus, same language slice)

Mirrors the original notebook's approach: interleave the per-language streams from
`bigcode/the-stack-dedup`, `.skip()` to the right offset for the current round, tokenize with the
**Qwen tokenizer** (not GPT-2 BPE), and write raw token ids to a per-round `.bin` file that's
deleted once the round is done training (keeps disk/page-cache pressure bounded across a long
session, same reasoning as the original). This runs once, single-process, in the parent notebook
kernel -- CPU-bound work, no reason to duplicate it across GPU worker processes.

`clean_code` only normalizes line endings -- Dockerfile/YAML/HCL are whitespace- and
indentation-sensitive, so nothing here collapses whitespace or strips content, unlike a
prose-cleaning function would.

In [ ]:
from datasets import load_dataset, interleave_datasets

def clean_code(text):
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    return text.strip("\n") + "\n"

_stream_state = {"iter": None, "next_row": 0}

def _open_iac_stream():
    lang_streams = [
        load_dataset(DATASET_NAME, data_dir=f"data/{lang}", split="train", streaming=True)
        for lang in IAC_LANGUAGES
    ]
    return interleave_datasets(lang_streams)

def _cleanup_earlier_rounds(round_idx):
    import glob, re
    for path in glob.glob(os.path.join(DATA_DIR, "round_*.bin")):
        m = re.search(r"round_(\d+)\.bin$", path)
        if m and int(m.group(1)) < round_idx:
            os.remove(path)

def tokenize_chunk(round_idx):
    """Streams this round's LINES_PER_CHUNK files, cleans + tokenizes them with the Qwen
    tokenizer, writes token ids to chunks/round_{round_idx}.bin as uint32 (Qwen's ~152k vocab
    does not fit in uint16). Returns the bin path -- the training subprocess reopens it itself
    rather than being handed an in-memory array."""
    os.makedirs(DATA_DIR, exist_ok=True)
    _cleanup_earlier_rounds(round_idx)
    bin_path = os.path.join(DATA_DIR, f"round_{round_idx}.bin")

    if not os.path.exists(bin_path):
        start_row = round_idx * LINES_PER_CHUNK
        if _stream_state["iter"] is None or _stream_state["next_row"] != start_row:
            stream = _open_iac_stream()
            if start_row > 0:
                stream = stream.skip(start_row)
            _stream_state["iter"] = iter(stream)

        eos_id = tokenizer.eos_token_id
        with open(bin_path, "wb") as f:
            count = 0
            for sample in _stream_state["iter"]:
                ids = tokenizer.encode(clean_code(sample["content"]))
                ids.append(eos_id)
                np.array(ids, dtype=np.uint32).tofile(f)
                count += 1
                if count >= LINES_PER_CHUNK:
                    break
        _stream_state["next_row"] = start_row + LINES_PER_CHUNK
        print(f"[round {round_idx}] files: {count}")

    n_tokens = os.path.getsize(bin_path) // np.dtype(np.uint32).itemsize
    print(f"[round {round_idx}] tokens: {n_tokens}")
    return bin_path


## Generate `train_worker.py`

Everything the training subprocess needs -- model name, paths, LoRA config, batching, and the LR
schedule -- gets baked into a standalone script from your **current** config cell values. Re-run
this cell any time you change something in "Sizing knobs to your VRAM" or the plan cell above it,
*before* your next Phase 1/Phase 2 run, so the script and the notebook don't drift out of sync.

Running training as `accelerate launch <script>` spawns a fresh, independent process per GPU via
`torchrun` -- not a fork of this Jupyter kernel -- which is what avoids the
`Cannot re-initialize CUDA in forked subprocess` failure entirely, regardless of what this kernel
has already touched.

In [ ]:
from string import Template

_SCRIPT_TEMPLATE = Template(r'''
import argparse
import math
import os

import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset
from accelerate import Accelerator
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, PeftModel

MODEL_NAME       = "$MODEL_NAME"
ADAPTER_DIR      = "$ADAPTER_DIR"
CKPT_DIR         = "$CKPT_DIR"
TRAIN_STATE_PATH = "$TRAIN_STATE_PATH"

BLOCK_SIZE                 = $BLOCK_SIZE
BATCH_SIZE                 = $BATCH_SIZE
GRAD_ACCUM_STEPS           = $GRAD_ACCUM_STEPS
USE_GRADIENT_CHECKPOINTING = $USE_GRADIENT_CHECKPOINTING

LORA_R       = $LORA_R
LORA_ALPHA   = $LORA_ALPHA
LORA_DROPOUT = $LORA_DROPOUT
LORA_TARGETS = $LORA_TARGETS

warmup_iters  = $warmup_iters
max_lr        = $max_lr
min_lr        = $min_lr
TOTAL_STEPS   = $TOTAL_STEPS
eval_interval = $eval_interval
eval_batches  = $eval_batches


class PackedTokenDataset(Dataset):
    def __init__(self, bin_path, block_size, split):
        data = np.memmap(bin_path, dtype=np.uint32, mode="r")
        cut = int(len(data) * 0.9)
        self.data = data[:cut] if split == "train" else data[cut:]
        self.block_size = block_size

    def __len__(self):
        return max(0, len(self.data) - self.block_size - 1)

    def __getitem__(self, i):
        x = torch.from_numpy(self.data[i:i + self.block_size].astype(np.int64))
        y = torch.from_numpy(self.data[i + 1:i + self.block_size + 1].astype(np.int64))
        return x, y


def create_loader(bin_path, block_size, batch_size, split):
    return DataLoader(
        PackedTokenDataset(bin_path, block_size, split),
        batch_size=batch_size,
        shuffle=True,
        pin_memory=True,
        num_workers=0,
    )


def build_optimizer(model):
    trainable = [p for p in model.parameters() if p.requires_grad]
    return torch.optim.AdamW(trainable, lr=max_lr, betas=(0.9, 0.95), eps=1e-8, weight_decay=0.01)


def get_lr(global_step):
    if global_step < warmup_iters:
        return max_lr * (global_step + 1) / warmup_iters
    if global_step > TOTAL_STEPS:
        return min_lr
    decay_ratio = (global_step - warmup_iters) / (TOTAL_STEPS - warmup_iters)
    return min_lr + 0.5 * (1.0 + math.cos(math.pi * decay_ratio)) * (max_lr - min_lr)


def save_checkpoint(unwrapped_model, optimizer, global_step, round_idx):
    os.makedirs(CKPT_DIR, exist_ok=True)
    unwrapped_model.save_pretrained(ADAPTER_DIR)
    torch.save({
        "optimizer_state_dict": optimizer.state_dict(),
        "global_step": global_step,
        "round_idx": round_idx,
    }, TRAIN_STATE_PATH)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--round_idx", type=int, required=True)
    parser.add_argument("--bin_path", type=str, required=True)
    parser.add_argument("--start_step", type=int, required=True)
    parser.add_argument("--end_step", type=int, required=True)
    parser.add_argument("--resume", action="store_true")
    args = parser.parse_args()

    accelerator = Accelerator(
        gradient_accumulation_steps=GRAD_ACCUM_STEPS,
        mixed_precision="fp16",
    )

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16)
    if USE_GRADIENT_CHECKPOINTING:
        base_model.gradient_checkpointing_enable()
        base_model.enable_input_require_grads()

    if args.resume:
        model = PeftModel.from_pretrained(base_model, ADAPTER_DIR, is_trainable=True)
    else:
        lora_config = LoraConfig(
            r=LORA_R,
            lora_alpha=LORA_ALPHA,
            lora_dropout=LORA_DROPOUT,
            target_modules=LORA_TARGETS,
            bias="none",
            task_type="CAUSAL_LM",
        )
        model = get_peft_model(base_model, lora_config)

    if accelerator.is_main_process:
        model.print_trainable_parameters()

    optimizer = build_optimizer(model)
    train_loader = create_loader(args.bin_path, BLOCK_SIZE, BATCH_SIZE, "train")
    val_loader = create_loader(args.bin_path, BLOCK_SIZE, BATCH_SIZE, "val")

    model, optimizer, train_loader, val_loader = accelerator.prepare(
        model, optimizer, train_loader, val_loader
    )

    if args.resume:
        state = torch.load(TRAIN_STATE_PATH, map_location="cpu")
        optimizer.load_state_dict(state["optimizer_state_dict"])

    @torch.no_grad()
    def estimate_loss():
        model.eval()
        out = {}
        for name, loader in [("train", train_loader), ("val", val_loader)]:
            losses = torch.zeros(eval_batches, device=accelerator.device)
            it = iter(loader)
            for k in range(eval_batches):
                try:
                    xb, yb = next(it)
                except StopIteration:
                    it = iter(loader)
                    xb, yb = next(it)
                result = model(input_ids=xb, labels=yb)
                losses[k] = accelerator.gather(result.loss.detach()).mean()
            out[name] = losses.mean().item()
        model.train()
        return out

    model.train()
    train_iter = iter(train_loader)
    logged_memory = False

    for step in range(args.start_step, args.end_step):
        lr = get_lr(step)
        for pg in optimizer.param_groups:
            pg["lr"] = lr

        if step % eval_interval == 0:
            losses = estimate_loss()
            if accelerator.is_main_process:
                print(f"step {step:6d} | lr {lr:.6f} | train {losses['train']:.4f} | val {losses['val']:.4f}")
            accelerator.wait_for_everyone()
            if accelerator.is_main_process:
                save_checkpoint(accelerator.unwrap_model(model), optimizer, step, args.round_idx)

        with accelerator.accumulate(model):
            try:
                xb, yb = next(train_iter)
            except StopIteration:
                train_iter = iter(train_loader)
                xb, yb = next(train_iter)

            out = model(input_ids=xb, labels=yb)
            accelerator.backward(out.loss)
            if accelerator.sync_gradients:
                accelerator.clip_grad_norm_(
                    [p for p in model.parameters() if p.requires_grad], 1.0
                )
            optimizer.step()
            optimizer.zero_grad()

        if not logged_memory and step == args.start_step:
            torch.cuda.synchronize(accelerator.device)
            allocated = torch.cuda.memory_allocated(accelerator.device) / 1e9
            reserved = torch.cuda.memory_reserved(accelerator.device) / 1e9
            print(f"[rank {accelerator.process_index}] GPU mem allocated: {allocated:.2f} GB "
                  f"/ reserved: {reserved:.2f} GB (of ~16 GB on this T4)")
            logged_memory = True

    accelerator.wait_for_everyone()
    if accelerator.is_main_process:
        save_checkpoint(accelerator.unwrap_model(model), optimizer, args.end_step, args.round_idx)
        print(f"round {args.round_idx} done, steps {args.start_step} -> {args.end_step}")


if __name__ == "__main__":
    main()
''')

script_content = _SCRIPT_TEMPLATE.substitute(
    MODEL_NAME=MODEL_NAME,
    ADAPTER_DIR=ADAPTER_DIR,
    CKPT_DIR=CKPT_DIR,
    TRAIN_STATE_PATH=TRAIN_STATE_PATH,
    BLOCK_SIZE=BLOCK_SIZE,
    BATCH_SIZE=BATCH_SIZE,
    GRAD_ACCUM_STEPS=GRAD_ACCUM_STEPS,
    USE_GRADIENT_CHECKPOINTING=USE_GRADIENT_CHECKPOINTING,
    LORA_R=LORA_R,
    LORA_ALPHA=LORA_ALPHA,
    LORA_DROPOUT=LORA_DROPOUT,
    LORA_TARGETS=repr(LORA_TARGETS),
    warmup_iters=warmup_iters,
    max_lr=max_lr,
    min_lr=min_lr,
    TOTAL_STEPS=TOTAL_STEPS,
    eval_interval=eval_interval,
    eval_batches=eval_batches,
)

with open(TRAIN_SCRIPT_PATH, "w") as f:
    f.write(script_content)

compile(script_content, TRAIN_SCRIPT_PATH, "exec")  # fail fast here, not mid-subprocess, if substitution broke something
print("wrote", TRAIN_SCRIPT_PATH)


## Helper: launch a training round as a subprocess

`accelerate launch` spawns `NUM_GPUS` independent OS processes running `train_worker.py`, each
bound to one GPU -- genuinely separate processes, not forks of this kernel. `check=True` makes a
failed round raise in the notebook instead of silently continuing.

In [ ]:
def run_training(round_idx, bin_path, start_step, end_step, resume):
    n = max(NUM_GPUS, 1)
    cmd = ["accelerate", "launch", "--num_processes", str(n), "--mixed_precision", "fp16"]
    if n > 1:
        cmd.append("--multi_gpu")
    cmd += [
        TRAIN_SCRIPT_PATH,
        "--round_idx", str(round_idx),
        "--bin_path", bin_path,
        "--start_step", str(start_step),
        "--end_step", str(end_step),
    ]
    if resume:
        cmd.append("--resume")
    print(" ".join(cmd))
    subprocess.run(cmd, check=True)

def read_train_state_cpu():
    """Peeks at global_step/round_idx without touching a GPU -- safe to call directly in
    this notebook kernel to plan the next round before launching the training subprocess."""
    state = torch.load(TRAIN_STATE_PATH, map_location="cpu")
    return state["global_step"], state["round_idx"]


## Phase 1 -- initial LoRA training (round 0)

Fresh adapter, fresh optimizer, `global_step` starts at 0. Trains on the first `LINES_PER_CHUNK`
files of the stream and saves to `ADAPTER_DIR` / `TRAIN_STATE_PATH` (written by the subprocess
itself, from its rank-0 process). Run this once, on day 1.

**After this cell finishes:** save `lora_checkpoints/` (adapter + train_state.pt) somewhere
durable -- e.g. upload it as a private Kaggle Dataset -- so Phase 2 can pick it back up in a fresh
session.

In [ ]:
round_idx = 0
bin_path = tokenize_chunk(round_idx)

print(f"Phase 1: LoRA fine-tuning round 0 across {max(NUM_GPUS, 1)} GPU(s)...")
run_training(round_idx, bin_path, start_step=0, end_step=STEPS_PER_ROUND, resume=False)


## Phase 2 -- continue training on the next chunk

Run this cell **once per session**, after Phase 1 has completed at least once. It:

1. Expects `CKPT_DIR` to already contain the adapter + `train_state.pt` from your last session --
   either still present from the same session, or restored from a Kaggle Dataset input mount by
   the bootstrap cell below.
2. Reads `global_step` / `round_idx` out of `train_state.pt` (on CPU, in the parent process) to
   know which round comes next automatically.
3. Tokenizes only the *next* chunk, then launches the training subprocess across both GPUs for
   exactly `STEPS_PER_ROUND` more steps, continuing the same `global_step` (so `get_lr` keeps
   decaying smoothly, no re-warmup), overwriting the checkpoint in place.
4. Stops once all `NUM_ROUNDS` are complete.

In [ ]:
# Bootstrap: if this is a fresh Kaggle session and CKPT_DIR is empty, restore from an
# uploaded Kaggle Dataset input mount (read-only) into the writable working directory.
if not os.path.exists(TRAIN_STATE_PATH) and os.path.exists(KAGGLE_INPUT_CKPT_DIR):
    os.makedirs(CKPT_DIR, exist_ok=True)
    shutil.copytree(os.path.join(KAGGLE_INPUT_CKPT_DIR, "adapter"), ADAPTER_DIR, dirs_exist_ok=True)
    shutil.copy(os.path.join(KAGGLE_INPUT_CKPT_DIR, "train_state.pt"), TRAIN_STATE_PATH)
    print(f"Restored checkpoint from {KAGGLE_INPUT_CKPT_DIR} -> {CKPT_DIR}")


In [ ]:
if not os.path.exists(TRAIN_STATE_PATH):
    raise FileNotFoundError(
        f"{TRAIN_STATE_PATH} not found. Upload the checkpoint you saved last time to "
        f"{KAGGLE_INPUT_CKPT_DIR} (as a Kaggle Dataset input) before running this cell."
    )

prev_global_step, _ = read_train_state_cpu()
round_idx = prev_global_step // STEPS_PER_ROUND

if round_idx >= NUM_ROUNDS:
    print(f"All {NUM_ROUNDS} rounds already completed (global_step={prev_global_step}). Nothing left to train.")
else:
    gc.collect()
    bin_path = tokenize_chunk(round_idx)

    start_step = prev_global_step
    end_step = (round_idx + 1) * STEPS_PER_ROUND
    print(f"Phase 2: continuing on round {round_idx} (steps {start_step} -> {end_step}) "
          f"across {max(NUM_GPUS, 1)} GPU(s)...")
    run_training(round_idx, bin_path, start_step=start_step, end_step=end_step, resume=True)
    print(f"Re-upload {CKPT_DIR} as a Kaggle Dataset to continue next session.")


## Quick generation check (single GPU, post-training)

Same prompt as the original from-scratch notebook's sanity-check cell, for a direct before/after
comparison against the 80M model's completions. Runs directly in this kernel on `cuda:0` -- fine
to touch CUDA here since it happens after training subprocesses have already exited, not before
one starts.

In [ ]:
import torch
from transformers import AutoModelForCausalLM
from peft import PeftModel

infer_device = "cuda:0" if torch.cuda.is_available() else "cpu"

base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16).to(infer_device)
infer_model = PeftModel.from_pretrained(base_model, ADAPTER_DIR).to(infer_device)
infer_model.eval()

prompt = "FROM python:3.11-slim\n\nWORKDIR /app\nCOPY requirements.txt "
inputs = tokenizer(prompt, return_tensors="pt").to(infer_device)

with torch.no_grad():
    generated = infer_model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=True,
        temperature=0.7,
        top_k=40,
        pad_token_id=tokenizer.pad_token_id,
    )
print(tokenizer.decode(generated[0], skip_special_tokens=True))


## Merge adapter + save final model (single GPU, post-training)

Folds the LoRA update into the base weights so downstream tooling doesn't need `peft` installed
just to run inference. Reuses `infer_model` from the generation-check cell above.

**Note on deploying this into `docker_auto_complete`'s existing backend:** `backend/inference/`
currently assumes the from-scratch `ModernTransformer` architecture and the GPT-2 tokenizer (see
`model/architecture.py`, `model/tokenizer.py`) with a ~50k vocab. Qwen2.5-Coder-0.5B is a
different architecture (standard Llama-style transformer) with a ~152k vocab, so it is **not** a
drop-in ONNX swap -- `backend/inference/model_service.py`, `backend/prompt/builder.py`, and
`model/tokenizer.py` would all need updating to match. Exporting to ONNX for that backend is a
separate follow-up step (e.g. via `optimum-cli export onnx`), not covered in this notebook.

In [ ]:
MERGED_DIR = "/kaggle/working/merged_model"

merged_model = infer_model.merge_and_unload()
merged_model.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_DIR)
print("Saved merged fp16 model to:", MERGED_DIR)
